In [1]:
import pandas as pd

df = pd.read_parquet("c_to_assembly.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5872 entries, 0 to 5871
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        5872 non-null   object
 1   c_code    5872 non-null   object
 2   assembly  5872 non-null   object
dtypes: object(3)
memory usage: 137.8+ KB


In [11]:
df.head(3)


,id,c_code,assembly,asm_len,c_len,assembly_clean
0,GPIO_LED_0357,"#include ""stm32f4xx_hal.h""\n\nvoid LED_Init(vo...",\nsample_0.o: file format elf32-littlearm\...,2533,593,"<func>\n<funcname> LED_Init\npush {<reg>, <reg..."
1,INTERRUPT_0453,"#include ""stm32f4xx_hal.h""\n\nvoid EXTI_Init(v...",\nsample_1.o: file format elf32-littlearm\...,2022,558,"<func>\n<funcname> EXTI_Init\npush {<reg>, <re..."
2,PWM_0409,"#include ""stm32f4xx_hal.h""\n\nTIM_HandleTypeDe...",\nsample_3.o: file format elf32-littlearm\...,4484,1212,"<func>\n<funcname> PWM_Init\npush {<reg>, <reg..."


In [12]:
print(df['assembly'][0])


sample_0.o:     file format elf32-littlearm


Disassembly of section .text:

00000000 <LED_Init>:
   0:	b580      	push	{r7, lr}
   2:	b086      	sub	sp, #24
   4:	af00      	add	r7, sp, #0
   6:	1d3b      	adds	r3, r7, #4
   8:	2200      	movs	r2, #0
   a:	601a      	str	r2, [r3, #0]
   c:	605a      	str	r2, [r3, #4]
   e:	609a      	str	r2, [r3, #8]
  10:	60da      	str	r2, [r3, #12]
  12:	611a      	str	r2, [r3, #16]
  14:	2300      	movs	r3, #0
  16:	603b      	str	r3, [r7, #0]
  18:	4b0e      	ldr	r3, [pc, #56]	@ (54 <LED_Init+0x54>)
  1a:	6b1b      	ldr	r3, [r3, #48]	@ 0x30
  1c:	4a0d      	ldr	r2, [pc, #52]	@ (54 <LED_Init+0x54>)
  1e:	f043 0310 	orr.w	r3, r3, #16
  22:	6313      	str	r3, [r2, #48]	@ 0x30
  24:	4b0b      	ldr	r3, [pc, #44]	@ (54 <LED_Init+0x54>)
  26:	6b1b      	ldr	r3, [r3, #48]	@ 0x30
  28:	f003 0310 	and.w	r3, r3, #16
  2c:	603b      	str	r3, [r7, #0]
  2e:	683b      	ldr	r3, [r7, #0]
  30:	f44f 5300 	mov.w	r3, #8192	@ 0x2000
  34:	607b      	str	r3, [r7, #4

In [13]:
df.isnull().sum()


id                0
c_code            0
assembly          0
asm_len           0
c_len             0
assembly_clean    0
dtype: int64

In [14]:
df["asm_len"] = df["assembly"].str.len()
df["c_len"] = df["c_code"].str.len()




In [15]:
import re
from collections import Counter

counter = Counter()

for asm in df["assembly"].sample(1000):
    for line in asm.splitlines():
        mnemonic = line.split()[0] if line else ""
        counter[mnemonic] += 1

counter.most_common(30)


[('', 6966),
 ('Disassembly', 1000),
 ('00000000', 1000),
 ('0:', 1000),
 ('2:', 1000),
 ('4:', 1000),
 ('6:', 1000),
 ('a:', 1000),
 ('c:', 1000),
 ('14:', 1000),
 ('18:', 1000),
 ('1a:', 1000),
 ('24:', 1000),
 ('26:', 1000),
 ('28:', 1000),
 ('3c:', 1000),
 ('10:', 992),
 ('1e:', 992),
 ('44:', 946),
 ('4c:', 946),
 ('36:', 944),
 ('52:', 939),
 ('34:', 938),
 ('50:', 936),
 ('2e:', 915),
 ('12:', 914),
 ('40:', 914),
 ('30:', 900),
 ('3e:', 900),
 ('22:', 897)]

In [47]:
import re

def split_functions(asm: str):
    """Split the assembly into functions based on addresses and function labels."""
    functions = []
    current = []

    for line in asm.splitlines():
        line = line.rstrip()

        # Function start line, e.g., 00000000 <UART_Init>:
        if re.match(r"^[0-9a-fA-F]+ <[^>]+>:", line):
            if current:
                functions.append("\n".join(current))
                current = []
            # Keep function name as first line
            func_name = re.findall(r"<([^>]+)>", line)
            if func_name:
                current.append(f"<funcname> {func_name[0]}")
            continue

        current.append(line)

    if current:
        functions.append("\n".join(current))

    return functions
def clean_instruction(line: str) -> str:
    line = line.strip()
    if not line:
        return ""


    # Remove address if present
    line = re.sub(r"^[0-9a-fA-F]+:\s*", "", line)

    # Remove leading opcode bytes (2–4 hex digits), stop at first mnemonic
    # Allow spaces/tabs but stop before an instruction mnemonic (letters)
    line = re.sub(r"^(?:[0-9a-fA-F]{2,4}\s+)+(?=[a-zA-Z])", "", line)

    # Remove objdump comments
    line = re.sub(r"\s*@.*$", "", line)

    # Normalize spacing
    line = re.sub(r"\s*,\s*", ", ", line)
    line = re.sub(r"\s+", " ", line)

    # Make sure we keep mnemonic even if operands are gone
    if not line.strip():
        return ""

    return line.strip()

def mask_registers_and_immediates(line: str) -> str:
    # Only mask registers in operand positions, keep mnemonics
    # e.g., add <reg>, <reg>, #<imm>
    line = re.sub(r"\b(r1[0-5]|r[0-9]|sp|lr|pc)\b", "<reg>", line)

    # Mask immediates (#0, #12, #0xFF)
    line = re.sub(r"#(?:0x[0-9a-fA-F]+|\d+)", "#<imm>", line)

    return line


def clean_function(fn_asm: str) -> str:
    """Clean a full function: remove metadata, mask registers/immediates, preserve mnemonics and HAL calls."""
    lines = []

    for line in fn_asm.splitlines():
        line = line.strip()
        if not line:
            continue

        # Skip metadata
        if "file format" in line.lower():
            continue
        if line.lower().startswith("disassembly of section"):
            continue
        if ".word" in line:
            continue

        line = clean_instruction(line)
        if not line:
            continue

        line = mask_registers_and_immediates(line)
        lines.append(line)

    if len(lines) < 2:
        return ""

    return "\n".join(lines)

def preprocess_assembly_keep_all(asm: str) -> str:
    """Preprocess the entire assembly: split into functions and clean each."""
    functions = split_functions(asm)
    cleaned_functions = []

    for fn in functions:
        cleaned = clean_function(fn)
        if cleaned:
            cleaned_functions.append(f"<func>\n{cleaned}\n</func>")

    return "\n\n".join(cleaned_functions)


In [48]:
df["assembly_clean"] = df["assembly"].apply(preprocess_assembly_keep_all)


In [50]:
print(df['assembly'][50])


sample_51.o:     file format elf32-littlearm


Disassembly of section .text:

00000000 <ADC_Init>:
   0:	b580      	push	{r7, lr}
   2:	b084      	sub	sp, #16
   4:	af00      	add	r7, sp, #0
   6:	463b      	mov	r3, r7
   8:	2200      	movs	r2, #0
   a:	601a      	str	r2, [r3, #0]
   c:	605a      	str	r2, [r3, #4]
   e:	609a      	str	r2, [r3, #8]
  10:	60da      	str	r2, [r3, #12]
  12:	4b1c      	ldr	r3, [pc, #112]	@ (84 <ADC_Init+0x84>)
  14:	4a1c      	ldr	r2, [pc, #112]	@ (88 <ADC_Init+0x88>)
  16:	601a      	str	r2, [r3, #0]
  18:	4b1a      	ldr	r3, [pc, #104]	@ (84 <ADC_Init+0x84>)
  1a:	f44f 3280 	mov.w	r2, #65536	@ 0x10000
  1e:	605a      	str	r2, [r3, #4]
  20:	4b18      	ldr	r3, [pc, #96]	@ (84 <ADC_Init+0x84>)
  22:	2200      	movs	r2, #0
  24:	609a      	str	r2, [r3, #8]
  26:	4b17      	ldr	r3, [pc, #92]	@ (84 <ADC_Init+0x84>)
  28:	2200      	movs	r2, #0
  2a:	611a      	str	r2, [r3, #16]
  2c:	4b15      	ldr	r3, [pc, #84]	@ (84 <ADC_Init+0x84>)
  2e:	2200      	movs	r2

In [51]:
print(df['assembly_clean'][50])

<func>
<funcname> ADC_Init
push {<reg>, <reg>}
sub <reg>, #<imm>
<reg>, <reg>, #<imm>
mov <reg>, <reg>
movs <reg>, #<imm>
str <reg>, [<reg>, #<imm>]
str <reg>, [<reg>, #<imm>]
str <reg>, [<reg>, #<imm>]
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
mov.w <reg>, #<imm>
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
movs <reg>, #<imm>
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
movs <reg>, #<imm>
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
movs <reg>, #<imm>
strb <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
movs <reg>, #<imm>
strb.w <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
movs <reg>, #<imm>
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
movs <reg>, #<imm>
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, #<imm>]
movs <reg>, #<imm>
str <reg>, [<reg>, #<imm>]
ldr <reg>, [<reg>, 

In [52]:
import pandas as pd

# Assume your DataFrame looks like this:
# df = pd.DataFrame({
#     'id': [...],
#     'c_code': [...],
#     'assembly': [...]  # preprocessed assembly
# })

# 1️⃣ Save as Parquet (efficient, columnar, fast)
df.to_parquet("preprocessed_dataset.parquet", engine="pyarrow", index=False)

# 2️⃣ Save as JSONL (one JSON per line, good for LLM training)
df.to_json("preprocessed_dataset.jsonl", orient="records", lines=True)
